# Inference Benchmark Analysis

**Author:** Abishek Bangalore Muralikrishna  
**Date:** May 2026  
**Model:** Meta Llama 3.1 8B Instruct  
**Framework:** TensorRT-LLM v0.21+ / NVIDIA NIM v1.15+  

---

## Overview

This notebook provides interactive analysis of benchmark results from the [inference-benchmarks](https://github.com/Abi5678/inference-benchmarks) project.

### Metrics Covered

| Metric | Description | Why It Matters |
|--------|-------------|----------------|
| **TTFT** (Time to First Token) | Latency from request to first response token | User-perceived latency |
| **TPOT** (Time per Output Token) | Average time between consecutive output tokens | Streaming quality |
| **Output Throughput** (tokens/s) | Total output tokens generated per second | System capacity |
| **Request Throughput** (req/s) | Number of requests completed per second | Concurrent user capacity |
| **Per-User Output Speed** (tps/user) | Speed experienced by each concurrent user | SLA compliance |
| **GPU Memory** | VRAM usage during inference | Cost & scaling |

In [ ]:
import json
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

COLORS = {
    'BF16': '#76B900',
    'FP8': '#E31937',
    'AWQ': '#00AEEF',
    'NIM': '#7B2D8E',
}

## 1. Load Results

In [ ]:
# Load all benchmark results
results = []
for f in glob.glob('../results/raw/*.json'):
    if 'summary' in f:
        continue
    try:
        with open(f) as fh:
            data = json.load(fh)
            data['_file'] = f
            results.append(data)
    except (json.JSONDecodeError, OSError):
        pass

print(f'Loaded {len(results)} benchmark results')

In [ ]:
# Parse into DataFrame
rows = []
for r in results:
    label = r.get('label', '')
    cfg = r.get('config', {})
    summary = r.get('summary', {})
    
    # Determine category
    if 'nim' in label:
        category = 'NIM'
    elif 'fp8' in label:
        category = 'FP8'
    elif 'awq' in label:
        category = 'AWQ'
    elif 'baseline' in label:
        category = 'BF16'
    elif 'buildflags' in label:
        category = 'BuildFlags'
    elif 'cuda_graph' in label:
        category = 'CUDAGraph'
    else:
        category = 'Other'
    
    rows.append({
        'label': label,
        'category': category,
        'batch_size': cfg.get('batch_size'),
        'concurrency': cfg.get('concurrency'),
        'isl': cfg.get('isl') or cfg.get('input_len', 128),
        'osl': cfg.get('osl') or cfg.get('output_len', 256),
        'streaming': cfg.get('streaming', True),
        'output_throughput_tps': r.get('output_throughput_tps'),
        'token_throughput_tps': r.get('token_throughput_tps'),
        'request_throughput_rps': r.get('request_throughput_rps'),
        'per_user_output_speed': r.get('per_user_output_speed'),
        'ttft_avg_ms': r.get('ttft_avg_ms') or summary.get('ttft_mean_ms'),
        'ttft_p50_ms': r.get('ttft_p50_ms') or summary.get('ttft_p50_ms'),
        'ttft_p99_ms': r.get('ttft_p99_ms') or summary.get('ttft_p99_ms'),
        'tpot_avg_ms': r.get('tpot_avg_ms') or r.get('tpot_ms') or summary.get('tpot_mean_ms'),
        'tpot_p50_ms': r.get('tpot_p50_ms'),
        'nim_throughput_tps': summary.get('throughput_tps'),
        'nim_ttft_p50_ms': summary.get('ttft_p50_ms'),
        'nim_tpot_mean_ms': summary.get('tpot_mean_ms'),
        'nim_latency_p99_ms': summary.get('latency_p99_ms'),
    })

df = pd.DataFrame(rows)
print(f'DataFrame: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Categories: {df["category"].value_counts().to_dict()}')
df.head()

## 2. Throughput Analysis

In [ ]:
# Throughput by quantization (batch size sweep)
batch_df = df[(df['concurrency'].isna()) & (df['category'].isin(['BF16', 'FP8', 'AWQ']))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cat, color in [('BF16', COLORS['BF16']), ('FP8', COLORS['FP8']), ('AWQ', COLORS['AWQ'])]:
    subset = batch_df[batch_df['category'] == cat].sort_values('batch_size')
    if subset.empty:
        continue
    axes[0].plot(subset['batch_size'], subset['output_throughput_tps'], 'o-', color=color, linewidth=2, label=cat)
    axes[1].plot(subset['batch_size'], subset['ttft_avg_ms'], 's--', color=color, linewidth=2, label=cat)

axes[0].set_xlabel('Batch Size')
axes[0].set_ylabel('Output Throughput (tokens/s)')
axes[0].set_title('Throughput vs Batch Size')
axes[0].legend()

axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('TTFT (ms)')
axes[1].set_title('TTFT vs Batch Size')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Throughput vs User Experience Tradeoff

In [ ]:
# This is THE key chart for production serving decisions
# X-axis: Per-user experience (how fast each user sees tokens)
# Y-axis: GPU utilization (total throughput)
# The tradeoff: more users → higher GPU utilization → worse per-user experience

fig, ax = plt.subplots(figsize=(10, 6))

for cat, color in [('BF16', COLORS['BF16']), ('FP8', COLORS['FP8'])]:
    # Use concurrency-sweep results
    conc_df = df[(df['category'] == cat) & (df['concurrency'].notna())].sort_values('concurrency')
    if conc_df.empty:
        continue
    
    # Combine batch and concurrency data
    all_data = pd.concat([conc_df])
    
    x = all_data['per_user_output_speed']
    y = all_data['output_throughput_tps']
    c = all_data['concurrency']
    
    valid = (x > 0) & (y > 0)
    ax.scatter(x[valid], y[valid], c=color, s=80, alpha=0.7, label=cat, edgecolors='white', linewidth=0.5)
    
    # Annotate concurrency levels
    for _, row in all_data[valid].iterrows():
        ax.annotate(f"c={int(row['concurrency'])}", (row['per_user_output_speed'], row['output_throughput_tps']),
                   fontsize=8, xytext=(5, 5), textcoords='offset points', color='gray')

# SLA reference lines
ax.axhline(y=5000, color='red', linestyle='--', alpha=0.3, label='SLA: 5000 gpu-tps')
ax.axvline(x=50, color='red', linestyle=':', alpha=0.3, label='Target: 50 tps/user')

ax.set_xlabel('Per-User Output Speed (tokens/user/s)')
ax.set_ylabel('Per-GPU Output Throughput (tokens/s)')
ax.set_title('Throughput vs User Experience Tradeoff\n(Higher GPU throughput + higher per-user speed = better)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. NIM Performance Under Load

In [ ]:
nim_df = df[df['category'] == 'NIM'].copy()
nim_conc = nim_df[nim_df['label'].str.contains('conc_')].copy()
nim_conc['conc_level'] = nim_df['label'].str.extract(r'conc_(\d+)').astype(float)
nim_conc = nim_conc.sort_values('conc_level')

if not nim_conc.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    axes[0,0].bar(range(len(nim_conc)), nim_conc['nim_throughput_tps'], color=COLORS['NIM'], alpha=0.85)
    axes[0,0].set_xticks(range(len(nim_conc)))
    axes[0,0].set_xticklabels([f'{int(c)}' for c in nim_conc['conc_level']])
    axes[0,0].set_xlabel('Concurrent Requests')
    axes[0,0].set_ylabel('Throughput (tokens/s)')
    axes[0,0].set_title('NIM Throughput vs Concurrency')
    
    axes[0,1].plot(nim_conc['conc_level'], nim_conc['nim_ttft_p50_ms'], 'o-', color=COLORS['NIM'], label='P50')
    axes[0,1].set_xlabel('Concurrent Requests')
    axes[0,1].set_ylabel('TTFT (ms)')
    axes[0,1].set_title('TTFT vs Concurrency')
    axes[0,1].legend()
    
    axes[1,0].plot(nim_conc['conc_level'], nim_conc['nim_tpot_mean_ms'], 'o-', color=COLORS['NIM'])
    axes[1,0].set_xlabel('Concurrent Requests')
    axes[1,0].set_ylabel('TPOT (ms)')
    axes[1,0].set_title('TPOT vs Concurrency')
    
    axes[1,1].plot(nim_conc['conc_level'], nim_conc['nim_latency_p99_ms'], 'o-', color='#E31937')
    axes[1,1].set_xlabel('Concurrent Requests')
    axes[1,1].set_ylabel('P99 Latency (ms)')
    axes[1,1].set_title('P99 End-to-End Latency')
    
    plt.tight_layout()
    plt.show()
else:
    print('No NIM concurrency results found. Run NIM benchmarks first.')

## 5. Build Flags Impact

In [ ]:
bf_df = df[df['category'] == 'BuildFlags'].copy()

if len(bf_df) >= 2:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    x = range(len(bf_df))
    tps = bf_df['output_throughput_tps'].fillna(0)
    tpot = bf_df['tpot_avg_ms'].fillna(0)
    
    ax2 = ax.twinx()
    
    bars = ax.bar(x, tps, color='#76B900', alpha=0.7, label='Throughput (tokens/s)')
    line = ax2.plot(x, tpot, 's-', color='#E31937', linewidth=2, label='TPOT (ms)')
    
    ax.set_xticks(x)
    ax.set_xticklabels(bf_df['label'], rotation=15, ha='right')
    ax.set_ylabel('Throughput (tokens/s)', color='#76B900')
    ax2.set_ylabel('TPOT (ms)', color='#E31937')
    ax.set_title('Build-Time Flags Impact')
    
    # Combined legend
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    
    plt.tight_layout()
    plt.show()
else:
    print('Not enough build flags data. Run: python scripts/run_benchmarks.py --phase build-flags')

## 6. Cost Analysis

Calculate cost-per-million-tokens based on GPU pricing.

In [ ]:
# GPU pricing (example: AWS)
GPU_PRICING = {
    'A10G': {'hourly': 1.006, 'memory_gb': 24, 'provider': 'AWS g5.xlarge'},
    'A100-40GB': {'hourly': 3.67, 'memory_gb': 40, 'provider': 'AWS p4d.24xlarge (1/8)'},
    'A100-80GB': {'hourly': 4.10, 'memory_gb': 80, 'provider': 'AWS p4d.24xlarge (1/8)'},
    'H100-80GB': {'hourly': 8.92, 'memory_gb': 80, 'provider': 'AWS p5.48xlarge (1/8)'},
}

# Example: Calculate cost for best BF16 config
best_bf16 = batch_df[batch_df['category'] == 'BF16'].nlargest(1, 'output_throughput_tps')

if not best_bf16.empty:
    tps = best_bf16['output_throughput_tps'].values[0]
    if tps and tps > 0:
        print('=== Cost per Million Tokens (BF16, best throughput) ===')
        print(f'  Throughput: {tps:.0f} tokens/s')
        for gpu, info in GPU_PRICING.items():
            tokens_per_hour = tps * 3600
            millions_per_hour = tokens_per_hour / 1_000_000
            cost_per_million = info['hourly'] / max(millions_per_hour, 0.001)
            print(f'  {gpu} ({info["provider"]}): ${cost_per_million:.2f}/M tokens')
else:
    print('No BF16 results available for cost analysis.')